# 09. 오픈소스에서 배우는 Jev 패턴 (3) 검색과 모델 라우팅

참고 프로젝트: [jev-search](https://github.com/superagents-lab/jev-search), [hermes-jev-skills](https://github.com/kerpopule/hermes-jev-skills)

## 배경

질문이 하나 들어왔을 때 답을 쓰기 전에 정해야 할 것들이 있습니다.

- 검색이 필요한가, 필요하면 어디서 찾을 것인가
- 얼마나 최근 자료가 필요한가
- 찾아온 결과 중 무엇을 읽을 것인가
- 어느 모델로 답할 것인가

이 결정들은 모두 정해진 option 중 하나를 고르는 일입니다. 그런데 보통은 큰 LLM 이 이것까지 다 합니다. jev-search 는 검색 소스 선택과 결과 관련도 판정을 Jev 에 맡기고, hermes-jev-skills 는 턴마다 어느 모델을 쓸지를 Jev 에 맡깁니다.

## 직관

도서관 안내 데스크입니다. 사서는 질문을 듣고 "그건 3층 논문 서가", "그건 신문 열람실"이라고 안내합니다. 책 내용을 설명해 주지는 않습니다. 어디로 갈지만 빠르게 정해 줍니다.

```
질의 ──▶ Jev 1회 호출 ─┬─ source      어느 소스에서 찾을까
                       ├─ time_range  얼마나 최근 자료가 필요한가
                       ├─ difficulty  어느 모델로 답할까
                       └─ rel_0..N    후보 결과 각각의 관련도
                       ▼
        고른 결과만 고른 모델에 넘겨 답변 작성 (LLM)
```

여기서는 실제 검색 엔진을 붙이지 않고, 후보 결과 다섯 개를 미리 정해 둡니다. 판단 부분만 보기 위해서입니다.

In [1]:
import sys
sys.path.insert(0, "..")  # 프로젝트 루트의 jev_agent 패키지를 불러오기 위함

import pandas as pd
from dotenv import load_dotenv
load_dotenv("../.env")

from jev_agent.jev import JevClient, choice, noul, score
from jev_agent.patterns import run_pattern

pd.set_option("display.max_colwidth", 60)
jev = JevClient()

In [2]:
def rows_table(output: dict) -> pd.DataFrame:
    """패턴 결과의 rows 를 표로 바꾼다. probability 는 큰 순서로 세 개까지 한 칸에 적는다."""
    records = []
    for item in output["rows"]:
        top = sorted(item["probabilities"].items(), key=lambda pair: -pair[1])[:3]
        records.append({
            "항목": item["title"],
            "판정": item["verdict"],
            "probability": ", ".join(f"{name} {value:.2f}" for name, value in top),
        })
    return pd.DataFrame(records)

## option 구성

검색 소스 option 은 일곱 개입니다. `no_search` 가 들어 있어서 검색이 필요 없다는 판단도 같은 질문으로 받습니다. jev-search 는 README 기준으로 12개 소스를 다루는데, 여기서는 종류별로 하나씩 묶어 단순하게 했습니다.

In [3]:
from jev_agent.patterns import routing

for name, description in routing.SOURCES.items():
    print(f"{name:13s} {description}")
print()
for index, item in enumerate(routing.SAMPLE_RESULTS):
    print(f"후보 {index}: {item['title']}")

web           일반 웹 검색. 제품 정보, 사용법, 폭넓은 주제.
news          뉴스 검색. 최근 사건, 발표, 출시 소식.
papers        논문 검색(arXiv). 연구 방법, 실험 결과, 학술 개념.
code          코드 검색(GitHub). 라이브러리 사용 예, 오류 메시지, 구현 코드.
community     커뮤니티 검색(Reddit, Hacker News). 사용 후기, 경험담, 의견.
encyclopedia  백과사전(Wikipedia). 정의, 역사, 인물, 기본 사실.
no_search     검색이 필요 없다. 인사이거나 이미 아는 내용으로 답할 수 있다.

후보 0: LangGraph interrupt 공식 문서
후보 1: 2026년 9월 TypeSafe, Jev 공개
후보 2: 파이썬 리스트 정렬 방법
후보 3: langgraph GitHub 이슈: interrupt 가 두 번 실행됨
후보 4: 서울 맛집 추천 10선


## 질의 하나 자세히 보기

In [4]:
query = routing.SAMPLE_QUERIES[0]
output = run_pattern(jev, "routing", {"query": query, "results": routing.SAMPLE_RESULTS})

print("질의:", query)
print({item["label"]: item["value"] for item in output["stats"]}, f"| {output['latency_ms']}ms | ${output['cost']:.6f}")
rows_table(output)

질의: LangGraph 에서 interrupt 후 재개하면 노드가 왜 다시 실행되나요?
{'검색 소스': 'code', '기간': 'past_year', '모델': '중간 모델', '쓸 결과': '1 / 5'} | 245ms | $0.000054


,항목,판정,probability
0,검색 소스,code,"code 0.93, web 0.07, community 0.00"
1,기간,past_year,"past_year 0.72, any 0.28, past_week 0.00"
2,모델 수준 0.93 / 2,중간 모델,"level_1 0.79, level_0 0.14, level_2 0.07"
3,langgraph GitHub 이슈: interrupt 가 두 번 실행됨,사용,relevance 0.77
4,LangGraph interrupt 공식 문서,제외,relevance 0.57
5,파이썬 리스트 정렬 방법,제외,relevance 0.00
6,서울 맛집 추천 10선,제외,relevance 0.00
7,"2026년 9월 TypeSafe, Jev 공개",제외,relevance 0.00


질문 여덟 개(소스, 기간, 난이도, 후보 다섯 개의 관련도)가 호출 한 번으로 끝났습니다. 관련도는 `score` 질문이라 0~2 사이 값이 나오고, 1.5 이상인 결과만 씁니다.

## 여러 질의 비교

In [5]:
records = []
for query in routing.SAMPLE_QUERIES:
    output = run_pattern(jev, "routing", {"query": query, "results": routing.SAMPLE_RESULTS})
    records.append({
        "질의": query,
        "소스": output["source"],
        "기간": output["stats"][1]["value"],
        "모델": output["model"],
        "쓸 결과": ", ".join(output["kept"]) or "(없음)",
        "지연(ms)": output["latency_ms"],
    })
pd.DataFrame(records)

,질의,소스,기간,모델,쓸 결과,지연(ms)
0,LangGraph 에서 interrupt 후 재개하면 노드가 왜 다시 실행되나요?,code,past_year,중간 모델,langgraph GitHub 이슈: interrupt 가 두 번 실행됨,308
1,이번 주에 나온 Jev 관련 소식 알려줘,news,past_week,작은 모델,"2026년 9월 TypeSafe, Jev 공개",248
2,트랜스포머 어텐션의 계산 복잡도를 줄이는 최근 연구는?,papers,past_year,중간 모델,(없음),241
3,안녕하세요,no_search,any,작은 모델,(없음),260


## 모델 라우팅을 실제로 적용하기

난이도 판정으로 채팅 모델을 고릅니다. 쉬운 질문에 큰 모델을 쓰지 않는 것만으로 비용이 크게 줄어듭니다. 아래에서는 세 단계에 OpenRouter 모델을 하나씩 대응시킵니다.

In [6]:
import time
from jev_agent.agent import build_chat_model

MODEL_IDS = {
    "작은 모델": "openai/gpt-5.4-nano",
    "중간 모델": "openai/gpt-5.4-mini",
    "큰 모델": "openai/gpt-5.4",
}

def answer_with_routing(query: str) -> dict:
    routed = run_pattern(jev, "routing", {"query": query, "results": routing.SAMPLE_RESULTS})
    model_id = MODEL_IDS[routed["model"]]
    started = time.perf_counter()
    reply = build_chat_model(model_id).invoke(query + "\n\n두 문장 이내로 답하세요.")
    return {
        "질의": query,
        "고른 모델": model_id,
        "Jev 지연(ms)": routed["latency_ms"],
        "LLM 지연(ms)": round((time.perf_counter() - started) * 1000),
        "답변": reply.content,
    }

pd.DataFrame([
    answer_with_routing("안녕하세요"),
    answer_with_routing("파이썬에서 리스트를 정렬하는 방법은?"),
    answer_with_routing("분산 락을 Redis 로 구현할 때 생기는 문제와 대안을 설계 관점에서 비교해 줘"),
])

,질의,고른 모델,Jev 지연(ms),LLM 지연(ms),답변
0,안녕하세요,openai/gpt-5.4-nano,242,932,안녕하세요! 무엇을 도와드릴까요?
1,파이썬에서 리스트를 정렬하는 방법은?,openai/gpt-5.4-nano,273,1018,"`list.sort()`로 리스트를 제자리 정렬하거나, `sorted(list)`로 새 리스트를 만들..."
2,분산 락을 Redis 로 구현할 때 생기는 문제와 대안을 설계 관점에서 비교해 줘,openai/gpt-5.4,274,4400,"Redis 분산 락은 단일 장애점, 네트워크 지연·클럭 불일치·TTL 만료로 인한 중복 획득, 장애 ..."


## 메시지 분류도 같은 방식

hermes-jev-skills 는 들어온 메시지를 분류하는 데도 Jev 를 씁니다. 얼마나 급한지, 어떤 종류인지, 사람이 봐야 하는지를 한 번에 묻습니다. 패턴 모듈 없이 `jev.decide` 로 바로 써 봅니다.

In [7]:
TRIAGE_QUESTIONS = {
    "urgency": score("이 메시지는 얼마나 급한가?", ["나중에 봐도 된다", "오늘 안에 봐야 한다", "지금 바로 봐야 한다"]),
    "kind": choice("이 메시지는 어떤 종류인가?", {
        "incident": "서비스 장애나 오류 보고",
        "question": "사용법이나 정보에 대한 질문",
        "request": "작업이나 변경 요청",
        "fyi": "참고용 공지나 공유",
    }),
    "needs_human": noul("사람이 직접 확인하고 결정해야 하는 메시지인가?"),
}
messages = [
    "결제 API 가 5분째 500 을 반환하고 있습니다. 주문이 전부 실패 중입니다.",
    "다음 주 수요일 팀 회식 장소 투표 부탁드려요.",
    "스테이징 DB 비밀번호 좀 바꿔 주실 수 있나요?",
    "참고로 어제 배포한 버전 릴리스 노트 공유합니다.",
]
records = []
for message in messages:
    result = jev.decide(message, TRIAGE_QUESTIONS)
    records.append({
        "메시지": message,
        "긴급도(0~2)": round(result["urgency"]["score"], 2),
        "종류": result["kind"]["choice"],
        "사람 확인": round(result["needs_human"]["noul"], 2),
        "지연(ms)": round(result.latency_ms),
    })
pd.DataFrame(records)

,메시지,긴급도(0~2),종류,사람 확인,지연(ms)
0,결제 API 가 5분째 500 을 반환하고 있습니다. 주문이 전부 실패 중입니다.,2.00,incident,0.69,308
1,다음 주 수요일 팀 회식 장소 투표 부탁드려요.,0.38,request,0.82,298
2,스테이징 DB 비밀번호 좀 바꿔 주실 수 있나요?,0.75,request,0.81,294
3,참고로 어제 배포한 버전 릴리스 노트 공유합니다.,0.17,fyi,0.56,328


## 정리

- 답을 쓰기 전의 결정(어디서 찾을지, 무엇을 읽을지, 어느 모델로 답할지)은 option 중 하나를 고르는 판단이라 Jev 에 맡길 수 있습니다.
- 질문이 여럿이어도 호출은 한 번이라, 라우팅을 추가해도 지연이 수백 ms 수준에 머뭅니다.
- 관련도 숫자는 모델의 판단이지 검증된 정확도가 아닙니다. jev-search README 도 이 점을 명시합니다. 중요한 용도라면 05번처럼 라벨 데이터로 직접 측정해야 합니다.
- 난이도 판정이 틀리면 어려운 질문이 작은 모델로 갑니다. 처음에는 기준을 보수적으로(큰 모델 쪽으로) 잡고, 로그를 보며 조정하는 것이 안전합니다.